##### Librerias

In [1]:
import xmlrpc.client # Comunicar con Odoo 
import pandas as pd
import base64, io, socket # Convierte Datos Binarios en texto
import numpy as np
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils import get_column_letter, column_index_from_string
import os
from dotenv import load_dotenv

##### Llaves

In [4]:
df_llaves = pd.read_excel("Bd_Llaves_Actualizado.xlsx", engine = "calamine")
ruta_carpeta = "C:/Users/HP/Desktop/DataPool/Proyecto_Transvial/Información_Proyecto_BI/"

##### Credenciales

In [34]:
load_dotenv() # Cargar las variables de entorno desde el archivo .env
url =  os.getenv("URL_ODOO")
username =  os.getenv("USER_JIM")
password = os.getenv("PASSWORD_JIM")
db = os.getenv("BD_ODOO")

##### Objecto para conectar ODOO + Interactuar ODOO

In [35]:
"""
    1. CONECTAR A ODOO - SE CREA OBJECTO PARA IDENTIFICAR + OBJECTO PARA HACER MODIFICACIONES
"""

# Crear objecto - Para conectar
common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")

# Identificarme
uid = common.authenticate(db, username, password, {})
print(f"uid conectado : {uid}")

# Crear objecto  - Para poder hacer consultas al modelo

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)

uid conectado : 20


In [36]:
"""
    2. VERIFICAR TIPO EMPRESA - SE CREA PARA VER EL CODIGO DE LA EMPRESA
"""

empresa = models.execute_kw(
    db, uid, password,
    "res.company", # Modelo o Tabla
    "search_read", # Metodo (Lo que quieres hacer)
    [],  # Sirve para filtrar por ejem [[["name", "=", "TATSA PERU S.A.C."]]],
    { #Opciones
        "fields": ["id", "name"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima"
        }
    }
)

empresa

[{'id': 4, 'name': 'ALCSA CORPORATION S.A.C.'},
 {'id': 6, 'name': 'AMAL SERVICES S.A.C.'},
 {'id': 3, 'name': 'MMLIMA S.A.C.'},
 {'id': 5, 'name': 'SOCSA CORP S.A.C.'},
 {'id': 2, 'name': 'TATSA PERU S.A.C.'},
 {'id': 1, 'name': 'TRANSVIAL LIMA SOCIEDAD ANONIMA CERRADA'}]

##### Datafreme Productos

In [8]:
"""
    1. CONEXION A DF 
"""
productos = models.execute_kw(
    db, uid, password,
    "product.template",
    "search_read",
    [
        [
            ["default_code", "!=", False]
        ]
    ],
    {
        "fields" : ["name", "numero_parte", "product_brand_id", "default_code",
                    "categ_id", "uom_id", "tipo_uso"],
        "context" :{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
            "search_default_consumable": 1
        }
    }
)

"""
    2. LIMPIEZA DF
"""

def convertir_many2one(valor):
    if isinstance(valor, list) and len(valor) == 2: # El isinstance pregunta si el valor es una lista.
        return valor[1]
    if valor is False:
        return None
    return valor


df_productos = pd.DataFrame(
    [
        {
            campo: convertir_many2one(fila.get(campo))
            for campo in [
            "name", 
            "numero_parte",
            "product_brand_id",
            "default_code",
            "categ_id",
            "uom_id",
            "tipo_uso"
            ]
        }
        for fila in productos
    ]
)

df_productos[["Familia", "Familia_2", "Familia_3"]] = (
    df_productos["categ_id"].str.split("/", n=2, expand=True)
)


df_productos = df_productos.rename(
    columns={
        "name" : "Producto",
        "numero_parte" : "Nro_Parte",
        "product_brand_id" : "Marca", 
        "default_code" : "Cod_Producto", 
        "uom_id": "Unidad_Medida", 
        "tipo_uso": "Uso"
    }
)

df_productos =  df_productos.drop(columns=["categ_id"])

df_productos = pd.merge(
    df_productos,
    df_llaves[["Cod_Producto", "Cod_Grupo_Sku", "Producto_Grupo"]],
    on="Cod_Producto",
    how="left"
)
df_productos["Cod_Grupo_Sku"] = df_productos["Cod_Grupo_Sku"].fillna(df_productos["Cod_Producto"])
df_productos["Producto_Grupo"] = df_productos["Producto_Grupo"].fillna(df_productos["Producto"])
df_productos = df_productos[["Cod_Grupo_Sku", "Cod_Producto", "Producto", "Producto_Grupo", "Nro_Parte", "Marca", 
                             "Unidad_Medida", "Familia", "Familia_2", "Familia_3", "Uso"]]

##### Dataframe Presupuesto Odoo

###### Ids Periodo Presupuesto

In [9]:
"""
    3. VERIFICAR EL ANIO FISCAL EN BASE AL TIPO DE EMPRESA
"""

anio_fiscal = models.execute_kw(
    db, uid, password,
    "account.fiscal.year", # Modelo
    "search_read", # Buscar y leer
    [
        #[
        #   ["company_id", "=", 1] # Para mas de un filtro ["company_id", "in", [1, 2]]
        #]
    ],
    {
        "fields": ["id", "name"],#, "company_id"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima"
        }
    }
)

anio_fiscal

[{'id': 1, 'name': '2023'},
 {'id': 2, 'name': '2024'},
 {'id': 12, 'name': '2025'},
 {'id': 14, 'name': '2026'},
 {'id': 16, 'name': '2027'},
 {'id': 18, 'name': '2028'},
 {'id': 20, 'name': '2029'},
 {'id': 22, 'name': '2030'},
 {'id': 23, 'name': '2031'},
 {'id': 25, 'name': '2032'},
 {'id': 26, 'name': '2039'}]

###### Presupuesto 2026 ODOO Tatsa

In [42]:
"""
    4. CREAR EL CONECTOR AL WIZARD PRIMERO EL FILTRO
"""

wizard_id = models.execute_kw(
    db, uid, password,
    "crossovered.budget.lines.wizardpre", #Wizard el modelo
    "create", # el metodo le indicas crear (search_read, create, read)
    [
        {
            "fiscal_year_id" : 14 # El año fiscal es 14 que en texto es 2026
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2] # La empresa es Tatsa con codigo 2
        }
    }
)

print("Wizard creado:", wizard_id)


"""
    5. AJECUTAR EL WIZARD CREADO
"""
accion = models.execute_kw(
    db, uid, password,
    "crossovered.budget.lines.wizardpre",
    "get_spresupuestos_oneperyear",
    [
        [wizard_id]
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
    
)


"""
    6. PARACTICAMENTE ES UN FILTRO - ESA ULTIMA LISTA CONTIENE LOS IDS QUE NECESITAMOS LEER
"""

ids = accion["domain"][0][2]

print("Cantidad de líneas:", len(ids))


"""
    7. TRAER EL INFORME CON ESOS FILTROS DEL WIZARD 
"""
datos_presupuesto = models.execute_kw(
    db, uid, password,
    "crossovered.budget.lines",
    "read",
    [ids],
    {
       "fields" : ["id", "crossovered_budget_id", "general_budget_id", "product_id", "analytic_account_id", "planned_amount_computed"],
       "context" : {
           "lang": "es_PE",
           "tz": "America/Lima",
           "allowed_company_ids": [2]
       }

    }
)

Wizard creado: 2165
Cantidad de líneas: 3984


In [43]:
"""
    8. FUNCION PARA MODIFICAR LA ESCTRUCTURA MANY2ONE A UN DF
"""
def convertir_many2one(valor):
    if isinstance(valor, list) and len(valor) == 2: # El isinstance pregunta si el valor es una lista.
        return valor[1]
    if valor is False:
        return None
    return valor


"""
    9. CREAR EL DF PRESUPUESTO USANDO LA FUNCION
"""

df_presupuesto_tat = pd.DataFrame(
    [
        {
            campo: convertir_many2one(fila.get(campo))
            for campo in [
                "id",
                "crossovered_budget_id",
                "general_budget_id",
                "product_id",
                "analytic_account_id",
                "planned_amount_computed"
            ]
        }
        for fila in datos_presupuesto
    ]
)


"""
    10. SE GENERA EL DF Y SE PROCEDE A LIMPIAR
"""
df_ppt_tat_2026 = df_presupuesto_tat.copy()
df_ppt_tat_2026 = df_ppt_tat_2026[["general_budget_id", "product_id", "analytic_account_id", "planned_amount_computed"]]
df_ppt_tat_2026 = df_ppt_tat_2026.rename(
    columns={
        "general_budget_id" : "Situación presupuestaria",
        "product_id" : "Cod_Producto",
        "analytic_account_id": "Cuenta Analítica",
        "planned_amount_computed":  "Presupuesto"
    }
)
df_ppt_tat_2026["Cod_Producto"] = df_ppt_tat_2026["Cod_Producto"].str.extract(r"\[(.*?)\]")
df_ppt_tat_2026["Periodo"] = 2026 #El presupuesto es del año 2026 por eso la clasifico de esa manera (Se debe cambiar segun el año)

df_ppt_tat_2026 = (
    df_ppt_tat_2026
    .groupby(
        ["Cod_Producto", "Cuenta Analítica", "Periodo"],
        as_index=False
    )
    .agg({
        "Situación presupuestaria": lambda x: ", ".join(
            x.dropna().astype(str).unique()
        ),
        "Presupuesto": "sum"
    })
)

###### Presupuesto ODOO Tvl

In [46]:
"""
    4. CREAR EL CONECTOR AL WIZARD PRIMERO EL FILTRO
"""

wizard_id = models.execute_kw(
    db, uid, password,
    "crossovered.budget.lines.wizardpre", #Wizard el modelo
    "create", # el metodo le indicas crear (search_read, create, read)
    [
        {
            "fiscal_year_id" : 14 # El año fiscal es 14 que en texto es 2026
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1] # La empresa es Transvial con codigo 1 
        }
    }
)

print("Wizard creado:", wizard_id)


"""
    5. AJECUTAR EL WIZARD CREADO
"""
accion = models.execute_kw(
    db, uid, password,
    "crossovered.budget.lines.wizardpre",
    "get_spresupuestos_oneperyear",
    [
        [wizard_id]
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
    
)


"""
    6. PARACTICAMENTE ES UN FILTRO - ESA ULTIMA LISTA CONTIENE LOS IDS QUE NECESITAMOS LEER
"""

ids = accion["domain"][0][2]

print("Cantidad de líneas:", len(ids))


"""
    7. TRAER EL INFORME CON ESOS FILTROS DEL WIZARD 
"""
datos_presupuesto = models.execute_kw(
    db, uid, password,
    "crossovered.budget.lines",
    "read",
    [ids],
    {
       "fields" : ["id", "crossovered_budget_id", "general_budget_id", "product_id", "analytic_account_id", "planned_amount_computed"],
       "context" : {
           "lang": "es_PE",
           "tz": "America/Lima",
           "allowed_company_ids": [1]
       }

    }
)

Wizard creado: 2166
Cantidad de líneas: 2056


In [47]:
"""
    8. FUNCION PARA MODIFICAR LA ESCTRUCTURA MANY2ONE A UN DF
"""
def convertir_many2one(valor):
    if isinstance(valor, list) and len(valor) == 2: # El isinstance pregunta si el valor es una lista.
        return valor[1]
    if valor is False:
        return None
    return valor


"""
    9. CREAR EL DF PRESUPUESTO USANDO LA FUNCION
"""

df_presupuesto_tvl = pd.DataFrame(
    [
        {
            campo: convertir_many2one(fila.get(campo))
            for campo in [
                "id",
                "crossovered_budget_id",
                "general_budget_id",
                "product_id",
                "analytic_account_id",
                "planned_amount_computed"
            ]
        }
        for fila in datos_presupuesto
    ]
)


"""
    10. SE GENERA EL DF Y SE PROCEDE A LIMPIAR
"""
df_ppt_tvl_2026 = df_presupuesto_tvl.copy()
df_ppt_tvl_2026 = df_ppt_tvl_2026[["general_budget_id", "product_id", "analytic_account_id", "planned_amount_computed"]]
df_ppt_tvl_2026 = df_ppt_tvl_2026.rename(
    columns={
        "general_budget_id" : "Situación presupuestaria",
        "product_id" : "Cod_Producto",
        "analytic_account_id": "Cuenta Analítica",
        "planned_amount_computed":  "Presupuesto"
    }
)
df_ppt_tvl_2026["Cod_Producto"] = df_ppt_tvl_2026["Cod_Producto"].str.extract(r"\[(.*?)\]")
df_ppt_tvl_2026["Periodo"] = 2026 #El presupuesto es del año 2026 por eso la clasifico de esa manera (Se debe cambiar segun el año)

df_ppt_tvl_2026 = (
    df_ppt_tvl_2026
    .groupby(
        ["Cod_Producto", "Cuenta Analítica", "Periodo"],
        as_index=False
    )
    .agg({
        "Situación presupuestaria": lambda x: ", ".join(
            x.dropna().astype(str).unique()
        ),
        "Presupuesto": "sum"
    })
)

##### Dataframe Registro de estado de Compras ODOO

###### Libreria de traduccion de estado de OC

In [12]:
"""
    1. SACAR EL DICCIONARIO PARA MODIFICAR ESTADO DE LA OC A ESPAÑOL
"""

fields_info = models.execute_kw(
    db,
    uid,
    password,
    "purchase.order",
    "fields_get",
    [],
    {
        "attributes": ["string", "type", "selection"]
    }
)

# Crear diccionario valor técnico -> etiqueta
estado_dict = dict(fields_info["state_new"]["selection"])

print(estado_dict)

{'draft': 'Petición Presupuesto', 'sent': 'Solicitud de Presupuesto enviada', 'to approve': 'Para Aprobar', 'purchase': 'Orden de Compra', 'done': 'Bloqueado', 'cancel': 'Cancelado', 'rechazado': 'Rechazado', 'cerrado': 'Cerrado', 'cerrado_no': 'Cerrada No Completada', 'parcialmente_cerrado': 'Cerrado Parcialmente'}


###### Dataframe de Tatsa Estado de OC

In [13]:
"""
    2. SACAR EL DATAFRAME PARA VER ESTADO DE LA ORDEN DE COMPRA
"""
estatus_oc_tat = models.execute_kw(
    db,
    uid,
    password,
    "purchase.order",
    "search_read",
    [
        [
            ["origin", "!=", False]
        ]
    ],
    {
        "fields": [
            "name",
            "origin",
            "state_new"
        ],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
            "quotation_only": True,
        },
        "order": "name ASC"
    }
)

"""
    3. LIMPIANDO EL DATAFRAME DEL ESTADO DE ORDEN DE COMPRA 
"""

df_estado_oc_tat = pd.DataFrame(estatus_oc_tat)
df_estado_oc_tat["state_new"] = df_estado_oc_tat["state_new"].map(estado_dict) #Cambiando el nombre de dato de bd por el nombre VISTO EN ODOO
df_estado_oc_tat = df_estado_oc_tat[["name", "origin", "state_new"]]
df_estado_oc_tat = df_estado_oc_tat.rename(columns={"name": "Cod_OC", "origin": "Cod_Rq", "state_new": "Estado_OC"})

###### Dataframe de Tvl Estado de OC

In [14]:
"""
    2. SACAR EL DATAFRAME PARA VER ESTADO DE LA ORDEN DE COMPRA
"""
estatus_oc_tvl = models.execute_kw(
    db,
    uid,
    password,
    "purchase.order",
    "search_read",
    [
        [
            ["origin", "!=", False]
        ]
    ],
    {
        "fields": [
            "name",
            "origin",
            "state_new"
        ],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1],
            "quotation_only": True,
        },
        "order": "name ASC"
    }
)

"""
    3. LIMPIANDO EL DATAFRAME DEL ESTADO DE ORDEN DE COMPRA 
"""

df_estado_oc_tvl = pd.DataFrame(estatus_oc_tvl)
df_estado_oc_tvl["state_new"] = df_estado_oc_tvl["state_new"].map(estado_dict) #Cambiando el nombre de dato de bd por el nombre VISTO EN ODOO
df_estado_oc_tvl = df_estado_oc_tvl[["name", "origin", "state_new"]]
df_estado_oc_tvl = df_estado_oc_tvl.rename(columns={"name": "Cod_OC", "origin": "Cod_Rq", "state_new": "Estado_OC"})

##### Dataframe Reporte de inventario ODOO

###### Para extraer el periodo del reporte de inventario

In [15]:
"""
    1. PARA EXTRAER EL ID DEL PERIODO DE WIZARD REPORTE DE INVENTARIO
"""

periodos = models.execute_kw(
    db, uid, password,
    "account.period.kardex",
    "search_read",
    [[]],
    {   
        "fields":["id", "code", "display_name"],
        "context": {
            "lang": "es_PE", 
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)

df_periodo_reporte_inv = pd.DataFrame(periodos) #El id es necesario 
df_periodo_reporte_inv[df_periodo_reporte_inv["code"]=="202612"]

,id,code,display_name
55,62,202612,202612


###### Reporte de Inventario Tatsa 2025

In [16]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 34, # Periodo 202501
            "fecha_final" : 45, # Periodo 202512
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv = pd.read_excel(archivo_en_memoria, dtype=str, header=9)

Wizard creado: 2185
Id creado del wizard 92382

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
1389264
b'PK'


C:\Users\HP\AppData\Local\Temp\ipykernel_19740\1023420091.py:89: Pandas4Warning: The default engine for reading 'xlsx' files will change from 'openpyxl' to 'calamine' in a future version. Pass engine='openpyxl' (or set the 'io.excel.xlsx.reader' option) to keep the current engine and silence this warning.
  df_inv = pd.read_excel(archivo_en_memoria, dtype=str, header=9)


In [17]:
"""
    5. LIMPIEZA DEL DATAFRAME DEL REPORTE DE INVENTARIO
"""

df_reporte_inv_tat_2025 = df_inv.copy()
df_reporte_inv_tat_2025["Periodo"] = 2025

###### Reporte de inventario Tatsa 2026

In [19]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 51, # Periodo 202501
            "fecha_final" : 62, # Periodo 202512
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv = pd.read_excel(archivo_en_memoria, dtype=str, header=9, engine="calamine")

Wizard creado: 2187
Id creado del wizard 92384

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
1590184
b'PK'


In [20]:
"""
    5. LIMPIEZA DEL DATAFRAME DEL REPORTE DE INVENTARIO
"""

df_reporte_inv_tat_2026 = df_inv.copy()
df_reporte_inv_tat_2026["Periodo"] = 2026

###### Reporte de inventario Tvl 2025

In [21]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 34, # Periodo 202501
            "fecha_final" : 45, # Periodo 202512
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv = pd.read_excel(archivo_en_memoria, dtype=str, header=9,  engine="calamine")

Wizard creado: 2188
Id creado del wizard 92385

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
1077740
b'PK'


In [22]:
"""
    5. LIMPIEZA DEL DATAFRAME DEL REPORTE DE INVENTARIO
"""

df_reporte_inv_tvl_2025 = df_inv.copy()
df_reporte_inv_tvl_2025["Periodo"] = 2025

###### Reporte de inventario Tvl 2026 

In [23]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 51, # Periodo 202601
            "fecha_final" : 62, # Periodo 202612
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv = pd.read_excel(archivo_en_memoria, dtype=str, header=9, engine="calamine")

Wizard creado: 2189
Id creado del wizard 92386

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
718684
b'PK'


In [24]:
"""
    5. LIMPIEZA DEL DATAFRAME DEL REPORTE DE INVENTARIO
"""

df_reporte_inv_tvl_2026 = df_inv.copy()
df_reporte_inv_tvl_2026["Periodo"] = 2026


##### Dataframe Reporte Kardex ODOO

###### Kardex Tatsa 2024 Odoo

In [25]:
"""
    1. DATOS DEL KARDEX (lo que vas a cambiar)
"""

FECHA_INICIO = "2024-01-01"      # AAAA-MM-DD
FECHA_FIN = "2024-12-31"

# Contexto del Kardex: lo mismo que ves en el Network, con 3 claves extra
# que le dicen a Odoo que es el Kardex VALORADO.
contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [2],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
# Son las que viste en tu captura: [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]
ubicaciones = [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]


"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)


"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)       # es el 280160 de tu captura (get_download_ls)


"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [2]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))


"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_crudo = pd.read_excel(archivo_en_memoria, dtype=str, header=8,  engine="calamine")

Wizard creado: 8246
Se proceso de 2024-01-01 al 2024-12-31. | Lineas procesadas: 7442 | Tiempo: 0 minutos 1 segundos
Id del adjunto: 290403
Kardex Valorado.xlsx
1616080
b'PK'


In [26]:
df_kardex_tatsa_2024 = df_kardex_crudo.copy()

###### Kardex Tatsa 2025 ODOO

In [27]:
"""
    1. DATOS DEL KARDEX (lo que vas a cambiar)
"""

FECHA_INICIO = "2025-01-01"      # AAAA-MM-DD
FECHA_FIN = "2025-12-31"

# Contexto del Kardex: lo mismo que ves en el Network, con 3 claves extra
# que le dicen a Odoo que es el Kardex VALORADO.
contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [2],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
# Son las que viste en tu captura: [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]
ubicaciones = [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]


"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)


"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)       # es el 280160 de tu captura (get_download_ls)


"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [2]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))


"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_crudo = pd.read_excel(archivo_en_memoria, dtype=str, header=8,  engine="calamine")

Wizard creado: 8247
Se proceso de 2025-01-01 al 2025-12-31. | Lineas procesadas: 66330 | Tiempo: 0 minutos 20 segundos
Id del adjunto: 290406
Kardex Valorado.xlsx
14385836
b'PK'


In [28]:
df_kardex_tatsa_2025 = df_kardex_crudo.copy()
df_kardex_tatsa_2025

,Fecha,Hora,Tipo,Serie,Número,Doc. Almacen,RUC,Proveedor,Supervisor,Tecnico,...,Importe,Cantidad.1,Importe.1,Cantidad.2,Importe.2,Ubicacion Origen,Ubicacion Destino,Almacen,IDProducto,Pedido
0,2025-06-19,16:59:27,NaN,NaN,NaN,TATRANSFATA00042,20492529789,TATSA PERU S.A.C.,NaN,NaN,...,1753.45263376667,0,0,1,1753.45263376667,ATATS/Mantenimiento,ATATS/Stock,ATATS/Stock,89248,NaN
1,2025-06-19,17:00:22,NaN,NaN,NaN,ATATOUTRAN00107,20492529789,TATSA PERU S.A.C.,NaN,Despacho Almacen,...,0,1,1753.45263376667,0,0,ATATS/Stock,PREST/Stock,ATATS/Stock,89248,NaN
2,2025-06-19,16:59:27,NaN,NaN,NaN,TATRANSFATA00042,20492529789,TATSA PERU S.A.C.,NaN,NaN,...,176.1276,0,0,2,176.1276,ATATS/Mantenimiento,ATATS/Stock,ATATS/Stock,90823,NaN
3,2025-06-19,17:00:22,NaN,NaN,NaN,ATATOUTRAN00107,20492529789,TATSA PERU S.A.C.,NaN,Despacho Almacen,...,0,2,176.1276,0,0,ATATS/Stock,PREST/Stock,ATATS/Stock,90823,NaN
4,2025-01-29,16:24:34,01,F001,00000113,ATAIVEDRSA02652,20612933228,CELIM HIDRAULICA SOLUCIONES INDUSTRIALES S.A.C.,NaN,NaN,...,141,0,0,3,141,Partner Locations/Vendors,ATATS/Stock,ATATS/Stock,94258,COMPR-00003646
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66326,2025-12-10,12:54:48,NaN,NaN,NaN,TATRANSFATA00277,20492529789,TATSA PERU S.A.C.,NaN,NaN,...,8490,0,0,3,8490,ATATS/Mantenimiento,AVCON/Stock,AVCON/Stock,95840,NaN
66327,2025-12-30,15:13:31,NaN,NaN,NaN,TATRANSFATA00313,20492529789,TATSA PERU S.A.C.,NaN,NaN,...,0,3,8490,0,0,AVCON/Stock,ATATS/Mantenimiento,AVCON/Stock,95840,NaN
66328,2025-12-10,12:54:48,NaN,NaN,NaN,TATRANSFATA00277,20492529789,TATSA PERU S.A.C.,NaN,NaN,...,4500,0,0,10,4500,ATATS/Mantenimiento,AVCON/Stock,AVCON/Stock,95841,NaN
66329,2025-12-30,15:13:31,NaN,NaN,NaN,TATRANSFATA00313,20492529789,TATSA PERU S.A.C.,NaN,NaN,...,0,10,4500,0,0,AVCON/Stock,ATATS/Mantenimiento,AVCON/Stock,95841,NaN


###### Kardex Tatsa 2026 ODOO

In [29]:
"""
    1. DATOS DEL KARDEX (lo que vas a cambiar)
"""

FECHA_INICIO = "2026-01-01"      # AAAA-MM-DD
FECHA_FIN = "2026-12-31"

# Contexto del Kardex: lo mismo que ves en el Network, con 3 claves extra
# que le dicen a Odoo que es el Kardex VALORADO.
contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [2],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
# Son las que viste en tu captura: [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]
ubicaciones = [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]


"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)


"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)       # es el 280160 de tu captura (get_download_ls)


"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [2]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))


"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_crudo = pd.read_excel(archivo_en_memoria, dtype=str, header=8,  engine="calamine")

Wizard creado: 8248
Se proceso de 2026-01-01 al 2026-12-31. | Lineas procesadas: 72271 | Tiempo: 0 minutos 19 segundos
Id del adjunto: 290407
Kardex Valorado.xlsx
15576080
b'PK'


In [30]:
df_kardex_tatsa_2026 = df_kardex_crudo.copy()

###### Kardex Tvl 2024 ODOO

In [31]:
"""
    1. DATOS DEL KARDEX (lo que vas a cambiar)
"""

FECHA_INICIO = "2024-01-01"      # AAAA-MM-DD
FECHA_FIN = "2024-12-31"

# Contexto del Kardex: lo mismo que ves en el Network, con 3 claves extra
# que le dicen a Odoo que es el Kardex VALORADO.
contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [1],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
# Son las que viste en tu captura: [39, 40, 44, 42, 41, 34, 192, 28, 8, 14, 15, 16]
ubicaciones = [39, 40, 44, 42, 41, 34, 192, 28, 8, 14, 15, 16]


"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)


"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)       # es el 280160 de tu captura (get_download_ls)


"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [1]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))


"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_crudo = pd.read_excel(archivo_en_memoria, dtype=str, header=8,  engine="calamine")

Wizard creado: 8249
Se proceso de 2024-01-01 al 2024-12-31. | Lineas procesadas: 91087 | Tiempo: 0 minutos 26 segundos
Id del adjunto: 290408
Kardex Valorado.xlsx
20813948
b'PK'


In [32]:
df_kardex_tvl_2024 = df_kardex_crudo.copy()

###### Kardex Tvl 2025 ODOO

In [37]:
"""
    1. DATOS DEL KARDEX (lo que vas a cambiar)
"""

FECHA_INICIO = "2025-01-01"      # AAAA-MM-DD
FECHA_FIN = "2025-12-31"

# Contexto del Kardex: lo mismo que ves en el Network, con 3 claves extra
# que le dicen a Odoo que es el Kardex VALORADO.
contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [1],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
# Son las que viste en tu captura: [39, 40, 44, 42, 41, 34, 192, 28, 8, 14, 15, 16]
ubicaciones = [39, 40, 44, 42, 41, 34, 192, 28, 8, 14, 15, 16]


"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)


"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)       # es el 280160 de tu captura (get_download_ls)


"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [1]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))


"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_crudo = pd.read_excel(archivo_en_memoria, dtype=str, header=8, engine="calamine")

Wizard creado: 8250
Se proceso de 2025-01-01 al 2025-12-31. | Lineas procesadas: 62670 | Tiempo: 0 minutos 16 segundos
Id del adjunto: 290422
Kardex Valorado.xlsx
13847152
b'PK'


In [38]:
df_kardex_tvl_2025 = df_kardex_crudo.copy()

###### Kardex Tvl 2026 Odoo

In [39]:
"""
    1. DATOS DEL KARDEX (lo que vas a cambiar)
"""

FECHA_INICIO = "2026-01-01"      # AAAA-MM-DD
FECHA_FIN = "2026-12-31"

# Contexto del Kardex: lo mismo que ves en el Network, con 3 claves extra
# que le dicen a Odoo que es el Kardex VALORADO.
contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [1],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
# Son las que viste en tu captura: [39, 40, 44, 42, 41, 34, 192, 28, 8, 14, 15, 16]
ubicaciones = [39, 40, 44, 42, 41, 34, 192, 28, 8, 14, 15, 16]


"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)


"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)       # es el 280160 de tu captura (get_download_ls)


"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [1]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))


"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_crudo = pd.read_excel(archivo_en_memoria, dtype=str, header=8,  engine="calamine")


Wizard creado: 8251
Se proceso de 2026-01-01 al 2026-12-31. | Lineas procesadas: 17264 | Tiempo: 0 minutos 4 segundos
Id del adjunto: 290423
Kardex Valorado.xlsx
3554084
b'PK'


In [40]:
df_kardex_tvl_2026 = df_kardex_crudo.copy()

##### Reporte de saldo disponible

###### REPORTE SALDO TATSA

In [44]:

"""
    Limpieza Periodo OC
"""
df_periodo_oc = pd.concat([
    df_reporte_inv_tat_2026[["SKU", "CENTRO DE COSTO", "ORDEN DE COMPRA", "Periodo"]],
    df_reporte_inv_tat_2025[["SKU", "CENTRO DE COSTO", "ORDEN DE COMPRA", "Periodo"]]
], ignore_index=True)

df_periodo_oc = df_periodo_oc[df_periodo_oc["ORDEN DE COMPRA"].notna()]
df_periodo_oc = df_periodo_oc.drop_duplicates()
df_periodo_oc = df_periodo_oc.rename(
    columns={"SKU": "Cod_Producto", "CENTRO DE COSTO": "Cuenta Analítica", "ORDEN DE COMPRA":"Cod_OC"})

"""
    Limpieza Kardex
"""
df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026.copy()
df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026_lp[["Sku", "Cantidad", "Importe", "Cantidad.1", "Importe.1", "Pedido"]]
df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026_lp[
    (df_kardex_tatsa_2026_lp["Pedido"].notna()) & (df_kardex_tatsa_2026_lp["Sku"].notna())]
df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026_lp.rename(
    columns={"Sku": "Cod_Producto", "Cantidad" : "Cant_Ingreso", "Importe" : "Monto_Ingreso", "Cantidad.1": "Cant_Salida", "Importe.1" : "Monto_Salida", "Pedido": "Cod_OC"})

df_kardex_tatsa_2026_lp["Cant_Ingreso"] = pd.to_numeric(df_kardex_tatsa_2026_lp["Cant_Ingreso"], errors="coerce")
df_kardex_tatsa_2026_lp["Monto_Ingreso"] = pd.to_numeric(df_kardex_tatsa_2026_lp["Monto_Ingreso"], errors="coerce")
df_kardex_tatsa_2026_lp["Cant_Salida"] = pd.to_numeric(df_kardex_tatsa_2026_lp["Cant_Salida"], errors="coerce")
df_kardex_tatsa_2026_lp["Monto_Salida"] = pd.to_numeric(df_kardex_tatsa_2026_lp["Monto_Salida"], errors="coerce")

df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026_lp.groupby(
    ["Cod_Producto", "Cod_OC"]
)[["Cant_Ingreso", "Monto_Ingreso", "Cant_Salida", "Monto_Salida"]].sum().reset_index()

df_kardex_tatsa_2026_lp["Cant_Ingreso_Fin"] = df_kardex_tatsa_2026_lp["Cant_Ingreso"] - df_kardex_tatsa_2026_lp["Cant_Salida"]
df_kardex_tatsa_2026_lp["Monto_Ingreso_Fin"] = df_kardex_tatsa_2026_lp["Monto_Ingreso"] - df_kardex_tatsa_2026_lp["Monto_Salida"]
df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026_lp[df_kardex_tatsa_2026_lp["Cant_Ingreso_Fin"] > 0]
df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026_lp.drop(columns=["Cant_Ingreso", "Monto_Ingreso", "Cant_Salida", "Monto_Salida"])


df_kardex_tatsa_2026_lp = pd.merge(
    df_kardex_tatsa_2026_lp, 
    df_periodo_oc,
    on=["Cod_Producto", "Cod_OC"],
    how='left'
)

df_kardex_tatsa_2026_lp = df_kardex_tatsa_2026_lp.groupby(
    ["Cod_Producto", "Cuenta Analítica", "Periodo"]
)[["Cant_Ingreso_Fin", "Monto_Ingreso_Fin"]].sum().reset_index()

df_kardex_tatsa_2026_lp["Monto_Ingreso_Fin"] = df_kardex_tatsa_2026_lp["Monto_Ingreso_Fin"]  * 1.18 

"""
    Limpiando Resporte de Inventario
"""

# -------------------- Numero de Requerimientos
df_rq_tatsa = df_reporte_inv_tat_2026.copy()

# seleccionar las columnas a trabajar
df_rq_tatsa = df_rq_tatsa[
    ["SKU", "REQ TATSA PERU S.A.C.", "ESTADO DE REQUERIMIENTO","CANTIDAD REQUERIDA","CENTRO DE COSTO", "ORDEN DE COMPRA", 
     "ESTADO DE ORDEN DE COMPRA","PROVEEDOR ELEGIDO","UNIDADES RFQ", "UNIDAD DE MEDIDA", "COSTO UNITARIO"]]

# cambio nombre a las columnas
df_rq_tatsa = df_rq_tatsa.rename(
    columns={"SKU": "Cod_Producto", "REQ TATSA PERU S.A.C.": "Cod_Rq", "ESTADO DE REQUERIMIENTO": "Estado_Rq", "CANTIDAD REQUERIDA": "Cant_Requerida",
             "CENTRO DE COSTO": "Centro_Costo", "ORDEN DE COMPRA": "Cod_OC","ESTADO DE ORDEN DE COMPRA": "Estado_Orden_Compra",
             "PROVEEDOR ELEGIDO": "Proveedor", "UNIDADES RFQ": "Cant_Requerida_Final", "UNIDAD DE MEDIDA": "Unidad_Medida", "COSTO UNITARIO": "Valor_Unitario"})

df_rq_tatsa["Cant_Requerida"] = pd.to_numeric(df_rq_tatsa["Cant_Requerida"], errors="coerce")
df_rq_tatsa["Cant_Requerida_Final"] = pd.to_numeric(df_rq_tatsa["Cant_Requerida_Final"], errors="coerce")
df_rq_tatsa["Valor_Unitario"] = pd.to_numeric(df_rq_tatsa["Valor_Unitario"], errors="coerce")


# filtrando df para que solo muestre requerimientos no en estado new
df_rq_tatsa =  df_rq_tatsa[df_rq_tatsa["Cod_Rq"] != "New"]

# Reemplazo de filas duplicadas de los G3 por repetir el mismo código en el RQ
filtroG3 = (df_rq_tatsa["Cod_Producto"].str.startswith('G3')) & (df_rq_tatsa["Cant_Requerida_Final"] > 0)
df_rq_tatsa.loc[filtroG3, "Cant_Requerida"] = df_rq_tatsa.loc[filtroG3, "Cant_Requerida_Final"]

# quitando filas duplicadas, solo me quedo con el requerimiento original sin los ingresos
df_rq_tatsa = df_rq_tatsa.drop_duplicates()

# trayendo el estado de la Orden_Compra forma de discriminir las OCs Duplicadas
df_rq_tatsa = pd.merge(df_rq_tatsa,
                       df_estado_oc_tat,
                       on=["Cod_OC","Cod_Rq"], #La unión es en base a dos columnas
                       how="left")

# mapeo trayendo la ocs pero solo por un registro
mapeo_estados = df_estado_oc_tat.drop_duplicates("Cod_OC").set_index("Cod_OC")["Estado_OC"]

df_rq_tatsa["Estado_OC"] = df_rq_tatsa["Estado_OC"].fillna(
    df_rq_tatsa["Cod_OC"].map(mapeo_estados))

#------------------------Cantidad Requerimiento recibido

df_rq_tatsa_recibido = df_reporte_inv_tat_2026.copy()

# seleccionando columnas a usar
df_rq_tatsa_recibido = df_rq_tatsa_recibido[
    ["SKU", "REQ TATSA PERU S.A.C.", "UNIDAD DE MEDIDA", "COSTO UNITARIO", "ORDEN DE COMPRA", "ALBARAN DE INGRESO", "CANTIDAD INGRESO"]]

# renombrando columna = 
df_rq_tatsa_recibido = df_rq_tatsa_recibido.rename(
    columns={"SKU": "Cod_Producto", "REQ TATSA PERU S.A.C.": "Cod_Rq", "ORDEN DE COMPRA": "Cod_OC",
             "UNIDAD DE MEDIDA": "Unidad_Medida", "COSTO UNITARIO": "Valor_Unitario", "ALBARAN DE INGRESO": "Albaran_Ingreso", 
             "CANTIDAD INGRESO": "Cant_Ingreso"})

df_rq_tatsa_recibido["Valor_Unitario"] = pd.to_numeric(df_rq_tatsa_recibido["Valor_Unitario"], errors="coerce")
df_rq_tatsa_recibido["Cant_Ingreso"] = pd.to_numeric(df_rq_tatsa_recibido["Cant_Ingreso"], errors="coerce")

df_rq_tatsa_recibido["Albaran_Ingreso"] = (
    df_rq_tatsa_recibido["Albaran_Ingreso"]
    .astype("string")
)

# creando columna de identificador
df_rq_tatsa_recibido["Delimitador"] = (
    df_rq_tatsa_recibido["Albaran_Ingreso"]
    .fillna("")
    .astype(str)
    .str[:10]
)

# creando una columna con las unidades de ingreso ajustada para que se igual a las requeridas por ejemplo: Cilindos a Litros, etc.
condiciones = [
    df_rq_tatsa_recibido["Cod_Producto"] == "G1070069",
    df_rq_tatsa_recibido["Unidad_Medida"] == "gal (EE. UU.) 3.78541",
    df_rq_tatsa_recibido["Unidad_Medida"] == "Cilindro"
    ]

resultados = [
    df_rq_tatsa_recibido["Cant_Ingreso"],
    df_rq_tatsa_recibido["Cant_Ingreso"] / 3.78541,
    df_rq_tatsa_recibido["Cant_Ingreso"] / 208.20,

]

df_rq_tatsa_recibido["Cant_Ingreso_Ajustada"] = np.select(condiciones, resultados, default=df_rq_tatsa_recibido["Cant_Ingreso"])

# ----------------------------------- Correcion de entregas en REQUERIMIENTO REAL
# correción de ingresos y asignado a la entrega principal

# 1. Separamos las devoluciones del resto de ingresos
es_devolucion = df_rq_tatsa_recibido["Albaran_Ingreso"].str.contains("TOTVENRSAA", na=False)
df_devoluciones = df_rq_tatsa_recibido[es_devolucion].copy()
df_ingresos_normales = df_rq_tatsa_recibido[~es_devolucion].copy()

# 2. Aplicamos la lógica estricta: si hay más de una, no conservamos NINGUNA.
# keep=False elimina todas las filas que tengan el subset duplicado.
df_devoluciones_validas = df_devoluciones.drop_duplicates(subset=["Cod_Producto", "Cod_Rq", "Cod_OC"], keep=False)

# 3. Unimos los ingresos con las devoluciones que pasaron el filtro de "únicas"
df_recibido_final = pd.concat([df_ingresos_normales, df_devoluciones_validas])

# Multiplicamos por -1 solo lo que es devolución
df_recibido_final["Cant_Net_Aux"] = np.where(
    df_recibido_final["Albaran_Ingreso"].str.contains("TOTVENRSAA", na=False),
    df_recibido_final["Cant_Ingreso_Ajustada"] * -1,
    df_recibido_final["Cant_Ingreso_Ajustada"]
)

# Sumamos por Producto y RQ para el cruce final
df_resumen_ingresos = df_recibido_final.groupby(["Cod_Producto", "Cod_Rq", "Cod_OC"])["Cant_Net_Aux"].sum().reset_index()
df_resumen_ingresos = df_resumen_ingresos.rename(columns={"Cant_Net_Aux": "Total_Cant_Recibida"})

df_rq_tatsa = pd.merge(
    df_rq_tatsa,
    df_resumen_ingresos,
    on=["Cod_Producto", "Cod_Rq", "Cod_OC"],
    how="left"
)

# Los que no tienen ingresos registrados quedan en 0
df_rq_tatsa["Total_Cant_Recibida"] = df_rq_tatsa["Total_Cant_Recibida"].fillna(0)

condiciones = [
    df_rq_tatsa["Cod_Producto"] == "G1070069",
    df_rq_tatsa["Unidad_Medida"] == "gal (EE. UU.) 3.78541",
    df_rq_tatsa["Unidad_Medida"] == "Cilindro"
]

resultados = [
    df_rq_tatsa["Total_Cant_Recibida"],
    round(df_rq_tatsa["Total_Cant_Recibida"],0),
    round(df_rq_tatsa["Total_Cant_Recibida"],0)
]

df_rq_tatsa["Total_Cant_Recibida"] = np.select(condiciones, resultados, default=df_rq_tatsa["Total_Cant_Recibida"])

# ajustando la cantidad requeridad del rq cuando es cerrado no completado
df_rq_tatsa["Cant_Requerida_Final_Ajustada"] = np.where(
    df_rq_tatsa["Estado_OC"] == "Cerrada No Completada",
    df_rq_tatsa["Total_Cant_Recibida"],
    df_rq_tatsa["Cant_Requerida_Final"]
)


# Reordenando las columnas 
df_rq_tatsa = df_rq_tatsa[['Cod_Producto', 'Cod_Rq','Estado_Rq', 'Cant_Requerida', 'Centro_Costo', 'Cod_OC', 'Estado_Orden_Compra', 
                           'Proveedor', 'Cant_Requerida_Final', 'Unidad_Medida', 'Valor_Unitario', 'Estado_OC', 
                           'Cant_Requerida_Final_Ajustada', 'Total_Cant_Recibida']]


# para ajustar los negativos de los G2 cuando el ingreso es mayor a las salidas
df_rq_tatsa["Total_Cant_Recibida"] = np.where(
    df_rq_tatsa["Cant_Requerida_Final_Ajustada"] - df_rq_tatsa["Total_Cant_Recibida"] < 0,
    df_rq_tatsa["Total_Cant_Recibida"] + (df_rq_tatsa["Cant_Requerida_Final_Ajustada"] - df_rq_tatsa["Total_Cant_Recibida"]),
    df_rq_tatsa["Total_Cant_Recibida"]
)



# agregando la columna de estado del Requerimiento = Pendiente, Parcial, Completo

condiciones = [
    df_rq_tatsa["Cod_OC"].isna(),
    df_rq_tatsa["Cant_Requerida_Final_Ajustada"] == df_rq_tatsa["Total_Cant_Recibida"],
    df_rq_tatsa["Total_Cant_Recibida"] == 0
]

resultados = [
    "",
    "Entregado",
    "Pendiente"
]
df_rq_tatsa["Estado_RQ_C/OC"] = np.select(condiciones, resultados, default="Parcial")

# agregando la columna llaves
df_rq_tatsa = pd.merge(
    df_rq_tatsa,
    df_llaves[["Cod_Producto", "Cod_Grupo_Sku"]],
    on = "Cod_Producto",
    how= "left"
)

# Limpieando valores en blancos
df_rq_tatsa["Cod_Grupo_Sku"] = df_rq_tatsa["Cod_Grupo_Sku"].fillna(df_rq_tatsa["Cod_Producto"])


# Precios Hitoricos
df_precio_historico  = pd.concat(
    [
        df_kardex_tatsa_2024[["Fecha","Sku", "Costo Promedio", "T. OP."]],
        df_kardex_tatsa_2025[["Fecha","Sku", "Costo Promedio", "T. OP."]],
        df_kardex_tatsa_2026[["Fecha","Sku", "Costo Promedio", "T. OP."]]
    ],
    ignore_index=True)

df_precio_historico["Costo Promedio"] = pd.to_numeric(df_precio_historico["Costo Promedio"], errors="coerce")
df_precio_historico["Fecha"] = pd.to_datetime(df_precio_historico["Fecha"],format="%Y-%m-%d", errors="coerce")

df_precio_historico = df_precio_historico[(df_precio_historico["T. OP."] == "02") & (df_precio_historico["Costo Promedio"]>0)]

df_precio_historico = df_precio_historico.rename(columns={"Sku": "Cod_Producto", "Costo Promedio" : "Valor_Unitario_Historico"})
df_precio_historico = pd.merge(
    df_precio_historico,
    df_llaves[["Cod_Producto", "Cod_Grupo_Sku"]],
    on="Cod_Producto",
    how="left"
)
df_precio_historico["Cod_Grupo_Sku"] = df_precio_historico["Cod_Grupo_Sku"].fillna(df_precio_historico["Cod_Producto"])

df_precio_historico = df_precio_historico.sort_values(
    "Fecha", ascending=False).drop_duplicates("Cod_Grupo_Sku")[["Cod_Grupo_Sku", "Fecha", "Valor_Unitario_Historico"]]



# ASIGNADO EL PRECIO HISTORICO 
df_rq_tatsa = pd.merge(
    df_rq_tatsa,
    df_precio_historico[["Cod_Grupo_Sku", "Valor_Unitario_Historico"]],
    on = "Cod_Grupo_Sku",
    how= "left"
)

# eliminado columna de apoyo
df_rq_tatsa = df_rq_tatsa.drop("Cod_Grupo_Sku", axis=1)


"""
    GENERANDO EL REPORTE FINAL DE SALDO DISPONIBLE
"""

#------ AGRUPANDO PRESUPUESTO + COMPRAS 2026 
df_saldos_tatsa = df_ppt_tat_2026.copy()
df_saldos_tatsa = pd.merge(
    df_saldos_tatsa, 
    df_kardex_tatsa_2026_lp,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

#------ AGRUPANDO PRESUPUESTO + RQ 
df_rq_tatsa_con_oc = df_rq_tatsa.copy()
df_rq_tatsa_con_oc = df_rq_tatsa_con_oc[df_rq_tatsa_con_oc["Estado_Orden_Compra"].notna()]
df_rq_tatsa_con_oc["RQ"] = (df_rq_tatsa_con_oc["Cant_Requerida_Final_Ajustada"] * df_rq_tatsa_con_oc["Valor_Unitario"] ) * 1.18
df_rq_tatsa_con_oc["Periodo"] = 2026
df_rq_tatsa_con_oc = df_rq_tatsa_con_oc.groupby(
    ["Cod_Producto", "Centro_Costo", "Periodo"]
)[["Cant_Requerida_Final_Ajustada", "RQ"]].sum().reset_index()
df_rq_tatsa_con_oc = df_rq_tatsa_con_oc.rename(columns={"Centro_Costo":"Cuenta Analítica"})

df_saldos_tatsa = pd.merge(
    df_saldos_tatsa, 
    df_rq_tatsa_con_oc,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

#------ AGRUPANDO PRESUPUESTO + RQ PENDIENTE CON OC

df_rq_tatsa_con_oc_pendiente = df_rq_tatsa.copy()
df_rq_tatsa_con_oc_pendiente = df_rq_tatsa_con_oc_pendiente[
    (df_rq_tatsa_con_oc_pendiente["Estado_Orden_Compra"].notna()) & (df_rq_tatsa_con_oc_pendiente["Estado_RQ_C/OC"].isin(["Parcial", "Pendiente"]))]
df_rq_tatsa_con_oc_pendiente["RQ_COC"] = (
    (df_rq_tatsa_con_oc_pendiente["Cant_Requerida_Final_Ajustada"] - df_rq_tatsa_con_oc_pendiente["Total_Cant_Recibida"]) * df_rq_tatsa_con_oc_pendiente["Valor_Unitario"] ) * 1.18
df_rq_tatsa_con_oc_pendiente["RQ_COC_und"] = (df_rq_tatsa_con_oc_pendiente["Cant_Requerida_Final_Ajustada"] - df_rq_tatsa_con_oc_pendiente["Total_Cant_Recibida"])

df_rq_tatsa_con_oc_pendiente["Periodo"] = 2026

df_rq_tatsa_con_oc_pendiente = df_rq_tatsa_con_oc_pendiente.groupby(
    ["Cod_Producto", "Centro_Costo", "Periodo"]
)[["RQ_COC_und", "RQ_COC"]].sum().reset_index()
df_rq_tatsa_con_oc_pendiente = df_rq_tatsa_con_oc_pendiente.rename(columns={"Centro_Costo":"Cuenta Analítica"})

df_saldos_tatsa = pd.merge(
    df_saldos_tatsa, 
    df_rq_tatsa_con_oc_pendiente,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

#------ AGRUPANDO PRESUPUESTO + RQ PENDIENTE SIN OC

df_rq_tatsa_sin_oc_pendiente = df_rq_tatsa.copy()
df_rq_tatsa_sin_oc_pendiente = df_rq_tatsa_sin_oc_pendiente[
    (df_rq_tatsa_sin_oc_pendiente["Estado_Orden_Compra"].isna()) & (df_rq_tatsa_sin_oc_pendiente["Estado_Rq"]=="SELECCIÓN DE LICITADOR")]
df_rq_tatsa_sin_oc_pendiente["RQ_SOC"] = (df_rq_tatsa_sin_oc_pendiente["Cant_Requerida"] * df_rq_tatsa_sin_oc_pendiente["Valor_Unitario_Historico"])* 1.18
df_rq_tatsa_sin_oc_pendiente["Periodo"] = 2026

df_rq_tatsa_sin_oc_pendiente = df_rq_tatsa_sin_oc_pendiente.groupby(
    ["Cod_Producto", "Centro_Costo", "Periodo"]
)[["Cant_Requerida", "RQ_SOC"]].sum().reset_index()
df_rq_tatsa_sin_oc_pendiente = df_rq_tatsa_sin_oc_pendiente.rename(columns={"Centro_Costo":"Cuenta Analítica"})

df_saldos_tatsa = pd.merge(
    df_saldos_tatsa, 
    df_rq_tatsa_sin_oc_pendiente,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

df_saldos_tatsa = df_saldos_tatsa.rename(
    columns= {
        "Cant_Ingreso_Fin" : "Compras_und",
        "Monto_Ingreso_Fin": "Compras",
        "Cant_Requerida_Final_Ajustada" : "RQ_und",
        "Cant_Requerida": "RQ_SOC_und"
    }
)

df_saldos_tatsa["Saldo"] = (
    df_saldos_tatsa["Presupuesto"].fillna(0) - (df_saldos_tatsa["Compras"].fillna(0) + df_saldos_tatsa["RQ_COC"].fillna(0) + df_saldos_tatsa["RQ_SOC"].fillna(0)))


df_saldos_tatsa = pd.merge(
    df_saldos_tatsa,
    df_productos[["Cod_Grupo_Sku", "Producto", "Cod_Producto", "Nro_Parte", "Marca", "Familia"]],
    on="Cod_Producto", 
    how="left"
)

df_saldos_tatsa = df_saldos_tatsa[
    ["Cod_Grupo_Sku", "Cod_Producto", "Producto", "Nro_Parte", "Marca", "Familia", "Cuenta Analítica", "Situación presupuestaria", "Periodo",
     "Presupuesto", "Compras", "Compras_und", "RQ", "RQ_und", "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Saldo"]]

###### REPORTE SALDO DISPONIBLE

In [48]:
"""
    Limpieza Periodo OC
"""
df_periodo_oc = pd.concat([
    df_reporte_inv_tvl_2026[["CODIGO SKU", "CENTRO DE COSTO", "ORDEN DE COMPRA", "Periodo"]],
    df_reporte_inv_tvl_2025[["CODIGO SKU", "CENTRO DE COSTO", "ORDEN DE COMPRA", "Periodo"]]
], ignore_index=True)

df_periodo_oc = df_periodo_oc[df_periodo_oc["ORDEN DE COMPRA"].notna()]
df_periodo_oc = df_periodo_oc.drop_duplicates()
df_periodo_oc = df_periodo_oc.rename(
    columns={"CODIGO SKU": "Cod_Producto", "CENTRO DE COSTO": "Cuenta Analítica", "ORDEN DE COMPRA":"Cod_OC"})

"""
    Limpieza Kardex
"""
df_kardex_tvl_2026_lp = df_kardex_tvl_2026.copy()
df_kardex_tvl_2026_lp = df_kardex_tvl_2026_lp[["Sku", "Cantidad", "Importe", "Cantidad.1", "Importe.1", "Pedido"]]
df_kardex_tvl_2026_lp = df_kardex_tvl_2026_lp[
    (df_kardex_tvl_2026_lp["Pedido"].notna()) & (df_kardex_tvl_2026_lp["Sku"].notna())]
df_kardex_tvl_2026_lp = df_kardex_tvl_2026_lp.rename(
    columns={"Sku": "Cod_Producto", "Cantidad" : "Cant_Ingreso", "Importe" : "Monto_Ingreso", "Cantidad.1": "Cant_Salida", "Importe.1" : "Monto_Salida", "Pedido": "Cod_OC"})

df_kardex_tvl_2026_lp["Cant_Ingreso"] = pd.to_numeric(df_kardex_tvl_2026_lp["Cant_Ingreso"], errors="coerce")
df_kardex_tvl_2026_lp["Monto_Ingreso"] = pd.to_numeric(df_kardex_tvl_2026_lp["Monto_Ingreso"], errors="coerce")
df_kardex_tvl_2026_lp["Cant_Salida"] = pd.to_numeric(df_kardex_tvl_2026_lp["Cant_Salida"], errors="coerce")
df_kardex_tvl_2026_lp["Monto_Salida"] = pd.to_numeric(df_kardex_tvl_2026_lp["Monto_Salida"], errors="coerce")

df_kardex_tvl_2026_lp = df_kardex_tvl_2026_lp.groupby(
    ["Cod_Producto", "Cod_OC"]
)[["Cant_Ingreso", "Monto_Ingreso", "Cant_Salida", "Monto_Salida"]].sum().reset_index()

df_kardex_tvl_2026_lp["Cant_Ingreso_Fin"] = df_kardex_tvl_2026_lp["Cant_Ingreso"] - df_kardex_tvl_2026_lp["Cant_Salida"]
df_kardex_tvl_2026_lp["Monto_Ingreso_Fin"] = df_kardex_tvl_2026_lp["Monto_Ingreso"] - df_kardex_tvl_2026_lp["Monto_Salida"]
df_kardex_tvl_2026_lp = df_kardex_tvl_2026_lp[df_kardex_tvl_2026_lp["Cant_Ingreso_Fin"] > 0]
df_kardex_tvl_2026_lp = df_kardex_tvl_2026_lp.drop(columns=["Cant_Ingreso", "Monto_Ingreso", "Cant_Salida", "Monto_Salida"])


df_kardex_tvl_2026_lp = pd.merge(
    df_kardex_tvl_2026_lp, 
    df_periodo_oc, # Reporte de inventario 2025 y 2026 TVL
    on=["Cod_Producto", "Cod_OC"],
    how='left'
)

df_kardex_tvl_2026_lp = df_kardex_tvl_2026_lp.groupby(
    ["Cod_Producto", "Cuenta Analítica", "Periodo"]
)[["Cant_Ingreso_Fin", "Monto_Ingreso_Fin"]].sum().reset_index()

df_kardex_tvl_2026_lp["Monto_Ingreso_Fin"] = df_kardex_tvl_2026_lp["Monto_Ingreso_Fin"]  * 1.18 


"""
    Limpiando Resporte de Inventario
"""

df_rq_tvl = df_reporte_inv_tvl_2026.copy()

# seleccionar las columnas a trabajar
df_rq_tvl = df_rq_tvl[
    ["CODIGO SKU", "REQ TRANSVIAL LIMA SOCIEDAD ANONIMA CERRADA", "ESTADO DE REQUERIMIENTO", "CANTIDAD REQUERIDA", "CENTRO DE COSTO", 
     "ORDEN DE COMPRA", "ESTADO DE ORDEN DE COMPRA", "PROVEEDOR ELEGIDO","UNIDADES RFQ", "UNIDAD DE MEDIDA.1", "COSTO UNITARIO"]]

# cambio nombre a las columnas
df_rq_tvl = df_rq_tvl.rename(
    columns={"CODIGO SKU": "Cod_Producto",
             "REQ TRANSVIAL LIMA SOCIEDAD ANONIMA CERRADA": "Cod_Rq",
             "ESTADO DE REQUERIMIENTO": "Estado_Rq",
             "CANTIDAD REQUERIDA": "Cant_Requerida",
             "CENTRO DE COSTO": "Cuenta Analítica",
             "ORDEN DE COMPRA": "Cod_OC",
             "ESTADO DE ORDEN DE COMPRA": "Estado_Orden_Compra",
             "PROVEEDOR ELEGIDO": "Proveedor",
             "UNIDADES RFQ": "Cant_Requerida_Final",
             "UNIDAD DE MEDIDA.1": "Unidad_Medida",
             "COSTO UNITARIO": "Valor_Unitario"})

df_rq_tvl["Cant_Requerida"] = pd.to_numeric(df_rq_tvl["Cant_Requerida"], errors="coerce")
df_rq_tvl["Cant_Requerida_Final"] = pd.to_numeric(df_rq_tvl["Cant_Requerida_Final"], errors="coerce")
df_rq_tvl["Valor_Unitario"] = pd.to_numeric(df_rq_tvl["Valor_Unitario"], errors="coerce")

# filtrando df para que solo muestre requerimientos no en estado new
df_rq_tvl =  df_rq_tvl[df_rq_tvl["Cod_Rq"] != "New"]

# Reemplazo de filas duplicadas de los G3 por repetir el mismo código en el RQ

filtroG3 = (df_rq_tvl["Cod_Producto"].str.startswith('G3')) & (df_rq_tvl["Cant_Requerida_Final"] > 0)
df_rq_tvl.loc[filtroG3, "Cant_Requerida"] = df_rq_tvl.loc[filtroG3, "Cant_Requerida_Final"]

# quitando filas duplicadas, solo me quedo con el requerimiento original sin los ingresos
df_rq_tvl = df_rq_tvl.drop_duplicates()

# trayendo el estado de la Orden_Compra forma de discriminir las OCs Duplicadas
df_rq_tvl = pd.merge(df_rq_tvl,
                     df_estado_oc_tvl,
                     on=["Cod_OC","Cod_Rq"], #La unión es en base a dos columnas
                     how="left")

# mapeo trayendo la ocs pero solo por un registro
mapeo_estados = df_estado_oc_tvl.drop_duplicates("Cod_OC").set_index("Cod_OC")["Estado_OC"]

df_rq_tvl["Estado_OC"] = df_rq_tvl["Estado_OC"].fillna(
    df_rq_tvl["Cod_OC"].map(mapeo_estados))

#------------------------Cantidad Requerimiento recibido

#-----------------------------------------------------------------------------------------------------------------------------------------
# creando el df de las cantidades recibidas
df_rq_tvl_recibido = df_reporte_inv_tvl_2026.copy()

# seleccionando columnas a usar
df_rq_tvl_recibido = df_rq_tvl_recibido[
    ["CODIGO SKU", "REQ TRANSVIAL LIMA SOCIEDAD ANONIMA CERRADA", "UNIDAD DE MEDIDA.1", "COSTO UNITARIO", "ORDEN DE COMPRA",
     "ALBARAN DE INGRESO", "CANTIDAD INGRESO"]]

# renombrando columna = 
df_rq_tvl_recibido = df_rq_tvl_recibido.rename(
    columns={"CODIGO SKU": "Cod_Producto",
             "REQ TRANSVIAL LIMA SOCIEDAD ANONIMA CERRADA": "Cod_Rq",
             "ORDEN DE COMPRA": "Cod_OC",
             "UNIDAD DE MEDIDA.1": "Unidad_Medida",
             "COSTO UNITARIO": "Valor_Unitario",
             "ALBARAN DE INGRESO": "Albaran_Ingreso",
             "CANTIDAD INGRESO": "Cant_Ingreso"})

df_rq_tvl_recibido["Valor_Unitario"] = pd.to_numeric(df_rq_tvl_recibido["Valor_Unitario"], errors="coerce")
df_rq_tvl_recibido["Cant_Ingreso"] = pd.to_numeric(df_rq_tvl_recibido["Cant_Ingreso"], errors="coerce")

df_rq_tvl_recibido["Albaran_Ingreso"] = (
    df_rq_tvl_recibido["Albaran_Ingreso"]
    .astype("string")
)

# creando una columna con las unidades de ingreso ajustada para que se igual a las requeridas por ejemplo: Cilindos a Litros, etc.
codigos_especiales = ["G1070069","G2080090","G2080062","G2080020","G2080129","G2080057","G2080049",
                      "G2070053", "G2080060","G2080002","G2080003","G2080021","G2080059","G2080091",
                      "G2080141"]

condiciones = [
    df_rq_tvl_recibido["Cod_Producto"].isin(codigos_especiales),
    df_rq_tvl_recibido["Unidad_Medida"] == "gal (EE. UU.) 3.78541",
    df_rq_tvl_recibido["Unidad_Medida"] == "Cilindro"
    ]

resultados = [
    df_rq_tvl_recibido["Cant_Ingreso"],
    df_rq_tvl_recibido["Cant_Ingreso"] / 3.78541,
    df_rq_tvl_recibido["Cant_Ingreso"] / 208.20,

]

df_rq_tvl_recibido["Cant_Ingreso_Ajustada"] = np.select(condiciones, resultados, default=df_rq_tvl_recibido["Cant_Ingreso"])

#-----------------------------------------------------------------------------------------------------------------------------------------------
# correción de ingresos y asignado a la entrega principal

# 1. Separamos las devoluciones del resto de ingresos
# Definimos los patrones de devoluciones en una variable para fácil mantenimiento
patron_devoluciones = "RNSOUENORA|AADTENRSAM"

# 1. Separamos las devoluciones del resto de ingresos usando el OR (|)
es_devolucion = df_rq_tvl_recibido["Albaran_Ingreso"].str.contains(patron_devoluciones, na=False)
df_devoluciones = df_rq_tvl_recibido[es_devolucion].copy()
df_ingresos_normales = df_rq_tvl_recibido[~es_devolucion].copy()

# 2. Aplicamos la lógica estricta: si hay más de una, no conservamos NINGUNA
df_devoluciones_validas = df_devoluciones.drop_duplicates(
    subset=["Cod_Producto", "Cod_Rq", "Cod_OC"], 
    keep=False
)

# 3. Unimos los ingresos con las devoluciones que pasaron el filtro
df_recibido_final = pd.concat([df_ingresos_normales, df_devoluciones_validas])

# 4. Multiplicamos por -1 si el albarán es cualquiera de los dos tipos de devolución
df_recibido_final["Cant_Net_Aux"] = np.where(
    df_recibido_final["Albaran_Ingreso"].str.contains(patron_devoluciones, na=False),
    df_recibido_final["Cant_Ingreso_Ajustada"] * -1,
    df_recibido_final["Cant_Ingreso_Ajustada"]
)

# 5. Agrupación y Cruce Final
df_resumen_ingresos = df_recibido_final.groupby(
    ["Cod_Producto", "Cod_Rq", "Cod_OC"]
)["Cant_Net_Aux"].sum().reset_index()

df_resumen_ingresos = df_resumen_ingresos.rename(columns={"Cant_Net_Aux": "Total_Cant_Recibida"})

df_rq_tvl = pd.merge(
    df_rq_tvl,
    df_resumen_ingresos,
    on=["Cod_Producto", "Cod_Rq", "Cod_OC"],
    how="left"
)

# Rellenar vacíos con 0
df_rq_tvl["Total_Cant_Recibida"] = df_rq_tvl["Total_Cant_Recibida"].fillna(0)

codigos_especiales = [
    "G1070069","G2080090","G2080062","G2080020","G2080129","G2080057","G2080049","G2070053",
    "G2080060","G2080002","G2080003","G2080021","G2080059","G2080091","G2080141"
]

condiciones = [
    df_rq_tvl["Cod_Producto"].isin(codigos_especiales),
    df_rq_tvl["Unidad_Medida"] == "gal (EE. UU.) 3.78541",
    df_rq_tvl["Unidad_Medida"] == "Cilindro"
]

resultados = [
    df_rq_tvl["Total_Cant_Recibida"],
    round(df_rq_tvl["Total_Cant_Recibida"],0),
    round(df_rq_tvl["Total_Cant_Recibida"],0)
]

df_rq_tvl["Total_Cant_Recibida"] = np.select(condiciones, resultados, default=df_rq_tvl["Total_Cant_Recibida"])

# ajustando la cantidad requeridad del rq cuando es cerrado no completado
df_rq_tvl["Cant_Requerida_Final_Ajustada"] = np.where(
    df_rq_tvl["Estado_OC"] == "Cerrada No Completada",
    df_rq_tvl["Total_Cant_Recibida"],
    df_rq_tvl["Cant_Requerida_Final"]
)

# Reordenando las columnas 
df_rq_tvl = df_rq_tvl[
    ['Cod_Producto', 'Cod_Rq', 'Estado_Rq', 'Cant_Requerida', 'Cuenta Analítica', 'Cod_OC', 'Estado_Orden_Compra', 'Proveedor', 
     'Cant_Requerida_Final', 'Unidad_Medida', 'Valor_Unitario', 'Estado_OC', 'Cant_Requerida_Final_Ajustada', 'Total_Cant_Recibida']]

# para ajustar los negativos de los G2 cuando el ingreso es mayor a las salidas
df_rq_tvl["Total_Cant_Recibida"] = np.where(
    df_rq_tvl["Cant_Requerida_Final_Ajustada"] - df_rq_tvl["Total_Cant_Recibida"] < 0,
    df_rq_tvl["Total_Cant_Recibida"] + (df_rq_tvl["Cant_Requerida_Final_Ajustada"] - df_rq_tvl["Total_Cant_Recibida"]),
    df_rq_tvl["Total_Cant_Recibida"]
)

# agregando la columna de estado del Requerimiento = Pendiente, Parcial, Completo

condiciones = [
    df_rq_tvl["Cod_OC"].isna(),
    df_rq_tvl["Cant_Requerida_Final_Ajustada"] == df_rq_tvl["Total_Cant_Recibida"],
    df_rq_tvl["Total_Cant_Recibida"] == 0
]

resultados = [
    "",
    "Entregado",
    "Pendiente"
]

df_rq_tvl["Estado_RQ_C/OC"] = np.select(condiciones, resultados, default="Parcial")

# agregando la columna llaves
df_rq_tvl = pd.merge(
    df_rq_tvl,
    df_llaves[["Cod_Producto", "Cod_Grupo_Sku"]],
    on = "Cod_Producto",
    how= "left"
)

# Limpieando valores en blancos
df_rq_tvl["Cod_Grupo_Sku"] = df_rq_tvl["Cod_Grupo_Sku"].fillna(df_rq_tvl["Cod_Producto"])

# Precios Hitoricos
df_precio_historico  = pd.concat(
    [
        df_kardex_tvl_2024[["Fecha","Sku", "Costo Promedio", "T. OP."]],
        df_kardex_tvl_2025[["Fecha","Sku", "Costo Promedio", "T. OP."]],
        df_kardex_tvl_2026[["Fecha","Sku", "Costo Promedio", "T. OP."]]
    ],
    ignore_index=True)

df_precio_historico["Costo Promedio"] = pd.to_numeric(df_precio_historico["Costo Promedio"], errors="coerce")
df_precio_historico["Fecha"] = pd.to_datetime(df_precio_historico["Fecha"],format="%Y-%m-%d", errors="coerce")

df_precio_historico = df_precio_historico[(df_precio_historico["T. OP."] == "02") & (df_precio_historico["Costo Promedio"]>0)]

df_precio_historico = df_precio_historico.rename(columns={"Sku": "Cod_Producto", "Costo Promedio" : "Valor_Unitario_Historico"})
df_precio_historico = pd.merge(
    df_precio_historico,
    df_llaves[["Cod_Producto", "Cod_Grupo_Sku"]],
    on="Cod_Producto",
    how="left"
)
df_precio_historico["Cod_Grupo_Sku"] = df_precio_historico["Cod_Grupo_Sku"].fillna(df_precio_historico["Cod_Producto"])

df_precio_historico = df_precio_historico.sort_values(
    "Fecha", ascending=False).drop_duplicates("Cod_Grupo_Sku")[["Cod_Grupo_Sku", "Fecha", "Valor_Unitario_Historico"]]



# ASIGNADO EL PRECIO HISTORICO 
df_rq_tvl = pd.merge(
    df_rq_tvl,
    df_precio_historico[["Cod_Grupo_Sku", "Valor_Unitario_Historico"]],
    on = "Cod_Grupo_Sku",
    how= "left"
)

# eliminado columna de apoyo
df_rq_tvl = df_rq_tvl.drop("Cod_Grupo_Sku", axis=1)


"""
    GENERANDO EL REPORTE FINAL DE SALDO DISPONIBLE
"""

#------ AGRUPANDO PRESUPUESTO + COMPRAS 2026 
df_saldos_tvl = df_ppt_tvl_2026.copy()
df_saldos_tvl = pd.merge(
    df_saldos_tvl, 
    df_kardex_tvl_2026_lp,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

#------ AGRUPANDO PRESUPUESTO + RQ 
df_rq_tvl_con_oc = df_rq_tvl.copy()
df_rq_tvl_con_oc = df_rq_tvl_con_oc[df_rq_tvl_con_oc["Estado_Orden_Compra"].notna()]
df_rq_tvl_con_oc["RQ"] = (df_rq_tvl_con_oc["Cant_Requerida_Final_Ajustada"] * df_rq_tvl_con_oc["Valor_Unitario"] ) * 1.18
df_rq_tvl_con_oc["Periodo"] = 2026
df_rq_tvl_con_oc = df_rq_tvl_con_oc.groupby(
    ["Cod_Producto", "Cuenta Analítica", "Periodo"]
)[["Cant_Requerida_Final_Ajustada", "RQ"]].sum().reset_index()

df_saldos_tvl = pd.merge(
    df_saldos_tvl, 
    df_rq_tvl_con_oc,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

#------ AGRUPANDO PRESUPUESTO + RQ PENDIENTE CON OC

df_rq_tvl_con_oc_pendiente = df_rq_tvl.copy()
df_rq_tvl_con_oc_pendiente = df_rq_tvl_con_oc_pendiente[
    (df_rq_tvl_con_oc_pendiente["Estado_Orden_Compra"].notna()) & (df_rq_tvl_con_oc_pendiente["Estado_RQ_C/OC"].isin(["Parcial", "Pendiente"]))]
df_rq_tvl_con_oc_pendiente["RQ_COC"] = (
    (df_rq_tvl_con_oc_pendiente["Cant_Requerida_Final_Ajustada"] - df_rq_tvl_con_oc_pendiente["Total_Cant_Recibida"]) * df_rq_tvl_con_oc_pendiente["Valor_Unitario"] ) * 1.18
df_rq_tvl_con_oc_pendiente["RQ_COC_und"] = (df_rq_tvl_con_oc_pendiente["Cant_Requerida_Final_Ajustada"] - df_rq_tvl_con_oc_pendiente["Total_Cant_Recibida"])

df_rq_tvl_con_oc_pendiente["Periodo"] = 2026

df_rq_tvl_con_oc_pendiente = df_rq_tvl_con_oc_pendiente.groupby(
    ["Cod_Producto", "Cuenta Analítica", "Periodo"]
)[["RQ_COC_und", "RQ_COC"]].sum().reset_index()

df_saldos_tvl = pd.merge(
    df_saldos_tvl, 
    df_rq_tvl_con_oc_pendiente,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

#------ AGRUPANDO PRESUPUESTO + RQ PENDIENTE SIN OC

df_rq_tvl_sin_oc_pendiente = df_rq_tvl.copy()
df_rq_tvl_sin_oc_pendiente = df_rq_tvl_sin_oc_pendiente[
    (df_rq_tvl_sin_oc_pendiente["Estado_Orden_Compra"].isna()) & (df_rq_tvl_sin_oc_pendiente["Estado_Rq"]=="SELECCIÓN DE LICITADOR")]
df_rq_tvl_sin_oc_pendiente["RQ_SOC"] = (df_rq_tvl_sin_oc_pendiente["Cant_Requerida"] * df_rq_tvl_sin_oc_pendiente["Valor_Unitario_Historico"])* 1.18
df_rq_tvl_sin_oc_pendiente["Periodo"] = 2026

df_rq_tvl_sin_oc_pendiente = df_rq_tvl_sin_oc_pendiente.groupby(
    ["Cod_Producto", "Cuenta Analítica", "Periodo"]
)[["Cant_Requerida", "RQ_SOC"]].sum().reset_index()


df_saldos_tvl = pd.merge(
    df_saldos_tvl, 
    df_rq_tvl_sin_oc_pendiente,
    on=["Cod_Producto", "Cuenta Analítica", "Periodo"],
    how="outer"
)

df_saldos_tvl = df_saldos_tvl.rename(
    columns= {
        "Cant_Ingreso_Fin" : "Compras_und",
        "Monto_Ingreso_Fin": "Compras",
        "Cant_Requerida_Final_Ajustada" : "RQ_und",
        "Cant_Requerida": "RQ_SOC_und"
    }
)

df_saldos_tvl["Saldo"] = (
    df_saldos_tvl["Presupuesto"].fillna(0) - (df_saldos_tvl["Compras"].fillna(0) + df_saldos_tvl["RQ_COC"].fillna(0) + df_saldos_tvl["RQ_SOC"].fillna(0)))


df_saldos_tvl = pd.merge(
    df_saldos_tvl,
    df_productos[["Cod_Grupo_Sku", "Producto", "Cod_Producto", "Nro_Parte", "Marca", "Familia"]],
    on="Cod_Producto", 
    how="left"
)

df_saldos_tvl = df_saldos_tvl[
    ["Cod_Grupo_Sku", "Cod_Producto", "Producto", "Nro_Parte", "Marca", "Familia", "Cuenta Analítica", "Situación presupuestaria", "Periodo",
     "Presupuesto", "Compras", "Compras_und", "RQ", "RQ_und", "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Saldo"]]

###### DANDO FORMATO AL EXCEL TVL Y TATSA

In [52]:
"""
    FORMATEANDO LISTA DE SALDOS DISPONIBLES TVL Y TATSA
"""
# ============================================================
# 1. CONFIGURACION GENERAL
# ============================================================

ARCHIVO = "Presupuesto_Saldos_02.xlsx"

# Columnas que deben mostrarse en soles
COLUMNAS_SOLES = ["J", "K", "M", "O", "Q", "S"]

# Columnas que deben tener el contenido centrado
COLUMNAS_CENTRADAS = [
    "A", "B", "I", "J", "K", "L", "M",
    "N", "O", "P", "Q", "R", "S"
]

# Anchos fijos; las demás columnas se ajustan al contenido
ANCHOS_FIJOS = {
    "C": 40,
    "F": 15,
    "G": 15,
    "H": 15,
    "A": 12,
    "B": 12,
    "D": 12,
    "E": 12,
    "I": 12,
}

# Colores de fondo de los encabezados
COLORES_ENCABEZADO = {
    "J": "41F828",
    "K": "FEE3DE",
    "L": "FEE3DE",
    "M": "D0F5FE",
    "N": "D0F5FE",
    "O": "D9D9D9",
    "P": "D9D9D9",
    "Q": "D9D9D9",
    "R": "D9D9D9",
    "S": "41F828",
}

FORMATO_SOLES = '"S/" #,##0.00'


# ============================================================
# 2. FUNCION PARA PREPARAR Y FORMATEAR UNA HOJA
# ============================================================

def formatear_hoja(df_original, nombre_hoja, writer):

    # A. Copiar el DataFrame para no modificar el original
    df = df_original.copy()

    # B. Convertir las columnas monetarias a numeros
    for letra in COLUMNAS_SOLES:

        # Excel empieza en 1; Python empieza en 0
        posicion = column_index_from_string(letra) - 1

        # Verificar que la columna exista en el DataFrame
        if posicion < len(df.columns):

            nombre_columna = df.columns[posicion]

            df[nombre_columna] = pd.to_numeric(
                df[nombre_columna],
                errors="coerce"
            )

    # C. Escribir los datos en la hoja correspondiente
    df.to_excel(
        writer,
        sheet_name=nombre_hoja,
        index=False
    )

    # D. Obtener la hoja de Excel creada
    ws = writer.sheets[nombre_hoja]

    # --------------------------------------------------------
    # E. Quitar las lineas de cuadricula
    # --------------------------------------------------------
    ws.sheet_view.showGridLines = False

    # --------------------------------------------------------
    # F. Aplicar fuente general a todas las celdas
    # --------------------------------------------------------
    fuente_general = Font(name="Consolas", size=8)

    for fila in ws.iter_rows():
        for celda in fila:
            celda.font = fuente_general

    # --------------------------------------------------------
    # G. Formatear el encabezado (fila 1)
    # --------------------------------------------------------
    lado = Side(style="thin")

    borde = Border(
        left=lado,
        right=lado,
        top=lado,
        bottom=lado
    )

    for celda in ws[1]:
        celda.font = Font(
            name="Consolas",
            size=8,
            bold=True
        )

        celda.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

        celda.border = borde

    # --------------------------------------------------------
    # H. Aplicar colores al encabezado
    # --------------------------------------------------------
    for letra, color in COLORES_ENCABEZADO.items():

        posicion = column_index_from_string(letra)

        if posicion <= ws.max_column:
            ws[f"{letra}1"].fill = PatternFill(
                start_color=color,
                end_color=color,
                fill_type="solid"
            )

    # --------------------------------------------------------
    # I. Centrar las columnas seleccionadas
    # --------------------------------------------------------
    for letra in COLUMNAS_CENTRADAS:

        posicion = column_index_from_string(letra)

        if posicion <= ws.max_column:
            for fila in range(1, ws.max_row + 1):

                celda = ws[f"{letra}{fila}"]

                celda.alignment = Alignment(
                    horizontal="center",
                    vertical="center"
                )

    # --------------------------------------------------------
    # J. Aplicar formato monetario desde la fila 2
    # --------------------------------------------------------
    for letra in COLUMNAS_SOLES:

        posicion = column_index_from_string(letra)

        if posicion <= ws.max_column:
            for fila in range(2, ws.max_row + 1):

                celda = ws[f"{letra}{fila}"]
                celda.number_format = FORMATO_SOLES

    # --------------------------------------------------------
    # K. Ajustar los anchos de las columnas
    # --------------------------------------------------------
    for i in range(1, ws.max_column + 1):

        letra = get_column_letter(i)

        # Si hay un ancho definido, usarlo
        if letra in ANCHOS_FIJOS:
            ws.column_dimensions[letra].width = (
                ANCHOS_FIJOS[letra]
            )

        else:
            # Buscar el contenido mas largo de la columna
            mayor = 0

            for fila in range(1, ws.max_row + 1):

                celda = ws[f"{letra}{fila}"]

                if celda.value is None:
                    continue

                if (
                    isinstance(celda.value, (int, float))
                    and letra in COLUMNAS_SOLES
                ):
                    largo = len(f"S/ {celda.value:,.2f}")

                else:
                    largo = len(str(celda.value))

                mayor = max(mayor, largo)

            # Agregar espacio adicional para que no quede ajustado
            ws.column_dimensions[letra].width = mayor + 2


# ============================================================
# 3. CREAR EL ARCHIVO CON LAS DOS HOJAS
# ============================================================

with pd.ExcelWriter(ARCHIVO, engine="openpyxl") as writer:

    formatear_hoja(
        df_saldos_tatsa,
        "TATSA",
        writer
    )

    formatear_hoja(
        df_saldos_tvl,
        "TVL",
        writer
    )


print("Archivo guardado:", ARCHIVO)

Archivo guardado: Presupuesto_Saldos_02.xlsx


In [53]:
with pd.ExcelWriter("Rq_Transito.xlsx", engine="openpyxl") as writer:
    df_rq_tatsa.to_excel(writer, sheet_name="TATSA", index=False),
    df_rq_tvl.to_excel(writer, sheet_name="TVL", index=False)